In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urldefrag

# Select webpages
pages = [
    "https://en.wikipedia.org/wiki/PageRank",
    "https://en.wikipedia.org/wiki/Search_engine",
    "https://en.wikipedia.org/wiki/Google",
    "https://en.wikipedia.org/wiki/Information_retrieval"
]

# Extract links using BeautifulSoup
graph = {}

for page in pages:
    response = requests.get(page, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(response.text, "html.parser")

    links = set()

    for a in soup.find_all("a", href=True):
        link = urldefrag(urljoin(page, a["href"]))[0]

        if link in pages:
            links.add(link)

    graph[page] = list(links)

# Initialize PageRank
N = len(graph)
d = 0.85
rank = {page: 1 / N for page in graph}

# Calculate PageRank
for i in range(100):
    new_rank = {page: (1 - d) / N for page in graph}

    dangling = sum(rank[p] for p in graph if not graph[p])

    for page, links in graph.items():
        for link in links:
            new_rank[link] += d * rank[page] / len(links)

    for page in graph:
        new_rank[page] += d * dangling / N

    difference = sum(abs(new_rank[p] - rank[p]) for p in graph)
    rank = new_rank

    if difference < 0.000001:
        break

# Display PageRank scores
print("\nPageRank Scores:")

for page, score in sorted(rank.items(), key=lambda x: x[1], reverse=True):
    print(page, ":", round(score, 4))


PageRank Scores:
https://en.wikipedia.org/wiki/PageRank : 0.3484
https://en.wikipedia.org/wiki/Search_engine : 0.3077
https://en.wikipedia.org/wiki/Google : 0.2003
https://en.wikipedia.org/wiki/Information_retrieval : 0.1436
